# Augmentation Showcase for Point Detection

Visualizing augmentation candidates for iguana point detection in drone imagery.

**Dataset properties:**
- Full images: 5472x3648, crops: 512x512
- Iguana visual size: ~20-40px
- 27% of iguanas have contrast < 10 vs background (heavily camouflaged)
- GT: FIDT heatmap with radius=2 at down_ratio=4

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import albumentations as A
from PIL import Image
from pathlib import Path

from animaloc.utils.augmentations import ObjectAwareRandomCrop

DATA = Path('/data/mnt/storage/Iguanas_From_Above/training_data/2026_04_07/2026_04_07_FMO03_02_05')
df = pd.read_csv(DATA / 'train/herdnet_format.csv')

# Pick an image with many iguanas
img_name = df['images'].value_counts().index[0]
img_full = np.array(Image.open(DATA / 'train/Default' / img_name))
annos = df[df['images'] == img_name]
all_kps = [(int(r['x']), int(r['y']), 0, 0) for _, r in annos.iterrows()]
print(f'{img_name}: {img_full.shape}, {len(all_kps)} iguanas')


In [ ]:
def get_crop(img, kps, seed=42):
    """Get a deterministic 512x512 crop with keypoints."""
    np.random.seed(seed)
    import random; random.seed(seed)
    crop = ObjectAwareRandomCrop(512, 512, min_edge_distance=0, p=1.0)
    t = A.Compose([crop], keypoint_params=A.KeypointParams(format='xy', remove_invisible=True))
    result = t(image=img, keypoints=kps)
    return result['image'], result['keypoints']

from IPython.display import display

def show_augmented(title, aug_list, img, kps, n_cols=4, n_samples=8):
    """Show multiple augmented versions of the same crop."""
    fig, axes = plt.subplots(2, n_cols, figsize=(4*n_cols, 8))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    transform = A.Compose(aug_list, keypoint_params=A.KeypointParams(format='xy', remove_invisible=True))
    
    for i, ax in enumerate(axes.flat):
        if i == 0:
            # Show original
            ax.imshow(img)
            for x, y, *_ in kps:
                ax.plot(x, y, 'r+', markersize=10, markeredgewidth=2)
            ax.set_title('Original', fontsize=10)
        else:
            result = transform(image=img, keypoints=kps)
            ax.imshow(np.clip(result['image'], 0, 255).astype(np.uint8) if result['image'].dtype != np.uint8 else result['image'])
            for x, y, *_ in result['keypoints']:
                ax.plot(x, y, 'r+', markersize=10, markeredgewidth=2)
            ax.set_title(f'Sample {i}', fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

# Get a good crop
crop_img, crop_kps = get_crop(img_full, all_kps, seed=10)
print(f'Crop: {crop_img.shape}, {len(crop_kps)} keypoints')
plt.figure(figsize=(5,5))
plt.imshow(crop_img)
for x, y, *_ in crop_kps:
    plt.plot(x, y, 'r+', markersize=12, markeredgewidth=2)
plt.title(f'Base crop ({len(crop_kps)} iguanas)')
plt.axis('off')


## Current Augmentations (already in pipeline)

In [ ]:
show_augmented('Current: Geometric (Flip + Rotate90 + ShiftScaleRotate)', [
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
], crop_img, crop_kps)

show_augmented('Current: Color (Brightness/Contrast + HSV + CLAHE)', [
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.5),
    A.CLAHE(clip_limit=4.0, p=0.3),
], crop_img, crop_kps)

show_augmented('Current: Blur (MotionBlur + Perspective)', [
    A.MotionBlur(p=0.5),
    A.Perspective(scale=0.05, p=0.3),
], crop_img, crop_kps)


## Proposed: Sensor Noise
Simulates different camera sensors, ISO settings, and compression artifacts.
Important because drone cameras vary and images are often JPEG-compressed.

In [ ]:
show_augmented('GaussNoise (sensor noise)', [
    A.GaussNoise(std_range=(0.03, 0.1), p=1.0),
], crop_img, crop_kps)

show_augmented('ISONoise (realistic camera noise)', [
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
], crop_img, crop_kps)

show_augmented('ImageCompression (JPEG artifacts)', [
    A.ImageCompression(quality_range=(40, 85), p=1.0),
], crop_img, crop_kps)


## Proposed: Resolution / Altitude Simulation
Simulates drone at different altitudes. Downscale+upscale makes iguanas smaller/blurrier.
Critical because the same model should work across flight altitudes.

In [ ]:
show_augmented('Downscale (higher altitude simulation)', [
    A.Downscale(scale_range=(0.4, 0.8), p=1.0),
], crop_img, crop_kps)

show_augmented('GaussianBlur (focus variation)', [
    A.GaussianBlur(blur_limit=(3, 9), p=1.0),
], crop_img, crop_kps)

show_augmented('Sharpen (over-sharpened images)', [
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0),
], crop_img, crop_kps)


## Proposed: Shadow & Lighting
Iguanas on volcanic rock — shadows from rocks, clouds, and time of day cause
dramatic local lighting changes. 27% of iguanas have contrast < 10 with background.

In [ ]:
show_augmented('PlasmaShadow (organic shadow patterns)', [
    A.PlasmaShadow(p=1.0),
], crop_img, crop_kps)

show_augmented('RandomShadow (directional shadows)', [
    A.RandomShadow(num_shadows_limit=(1, 3), shadow_dimension=5, p=1.0),
], crop_img, crop_kps)

show_augmented('RandomGamma (exposure variation)', [
    A.RandomGamma(gamma_limit=(60, 140), p=1.0),
], crop_img, crop_kps)

show_augmented('RandomToneCurve (non-linear lighting)', [
    A.RandomToneCurve(scale=0.15, p=1.0),
], crop_img, crop_kps)


## Proposed: Occlusion & Dropout
Simulates partial occlusion by vegetation, rocks, or other objects.
Forces the model to detect iguanas from partial views.
**Important for point detection**: the model should still predict the point
even when part of the animal is occluded.

In [ ]:
show_augmented('CoarseDropout (random occlusion patches)', [
    A.CoarseDropout(num_holes_range=(5, 15), hole_height_range=(10, 40), hole_width_range=(10, 40), p=1.0),
], crop_img, crop_kps)

show_augmented('PixelDropout (sparse pixel noise)', [
    A.PixelDropout(dropout_prob=0.02, p=1.0),
], crop_img, crop_kps)


## Proposed: Color Space Variation
Different cameras produce different color responses.
RGBShift and ChannelShuffle teach color invariance.

In [ ]:
show_augmented('RGBShift (camera white balance variation)', [
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=1.0),
], crop_img, crop_kps)

show_augmented('ChannelShuffle (extreme color invariance)', [
    A.ChannelShuffle(p=1.0),
], crop_img, crop_kps)

show_augmented('FancyPCA (ImageNet-style color aug)', [
    A.FancyPCA(alpha=0.1, p=1.0),
], crop_img, crop_kps)


## Proposed: Weather Simulation
Galapagos weather: fog, haze, occasional rain.

In [ ]:
show_augmented('RandomFog (haze/mist)', [
    A.RandomFog(fog_coef_range=(0.1, 0.4), alpha_coef=0.1, p=1.0),
], crop_img, crop_kps)

show_augmented('RandomSunFlare (lens flare from sun)', [
    A.RandomSunFlare(src_radius=100, p=1.0),
], crop_img, crop_kps)


## Proposed: Copy-Paste (Mosaic)
Combine crops from different images. This is very powerful for point detection because:
1. Increases effective scene diversity (critical when training from only 19 images)
2. Places iguanas in novel backgrounds they weren't originally photographed in
3. The albumentations Mosaic creates a 2x2 grid from 4 crops

In [ ]:
# Manual mosaic demonstration (albumentations Mosaic needs special metadata pipeline)
# Here we show the concept: 4 random crops stitched into a 2x2 grid, then cropped to 512x512

import random

def make_mosaic(img_full, df, img_dir, crop_size=512):
    """Create a 2x2 mosaic from 4 random crops, then center-crop to crop_size."""
    crops = []
    all_mosaic_kps = []
    
    # Pick 4 random images and crop each
    for i in range(4):
        img_name = random.choice(df['images'].unique())
        img = np.array(Image.open(img_dir / img_name))
        kps = [(int(r['x']), int(r['y']), 0, 0) for _, r in df[df['images'] == img_name].iterrows()]
        
        c_img, c_kps = get_crop(img, kps, seed=random.randint(0, 10000))
        crops.append(c_img)
        
        # Offset keypoints for mosaic position
        ox = (i % 2) * crop_size
        oy = (i // 2) * crop_size
        for x, y, *rest in c_kps:
            all_mosaic_kps.append((x + ox, y + oy))
    
    # Assemble 2x2 grid (1024x1024)
    grid = np.zeros((crop_size*2, crop_size*2, 3), dtype=np.uint8)
    grid[:crop_size, :crop_size] = crops[0]
    grid[:crop_size, crop_size:] = crops[1]
    grid[crop_size:, :crop_size] = crops[2]
    grid[crop_size:, crop_size:] = crops[3]
    
    # Random center crop back to 512x512
    cx = random.randint(0, crop_size)
    cy = random.randint(0, crop_size)
    final = grid[cy:cy+crop_size, cx:cx+crop_size]
    
    # Adjust keypoints
    final_kps = [(x-cx, y-cy) for x, y in all_mosaic_kps 
                 if 0 <= x-cx < crop_size and 0 <= y-cy < crop_size]
    
    return final, final_kps, grid

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Copy-Paste / Mosaic: 4 crops combined into new training samples', fontsize=14, fontweight='bold')

img_dir = DATA / 'train/Default'
for i in range(4):
    final, final_kps, grid = make_mosaic(img_full, df, img_dir)
    
    # Show full 2x2 grid
    axes[0, i].imshow(grid)
    axes[0, i].set_title(f'2x2 grid {i+1}', fontsize=10)
    axes[0, i].axis('off')
    
    # Show final crop
    axes[1, i].imshow(final)
    for x, y in final_kps:
        axes[1, i].plot(x, y, 'r+', markersize=10, markeredgewidth=2)
    axes[1, i].set_title(f'Crop ({len(final_kps)} iguanas)', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()


## Proposed: Combined Pipeline
Recommended augmentation stack for point detection with stitcher inference.

In [ ]:
proposed_pipeline = [
    # --- Geometric (keypoint-aware) ---
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.15, rotate_limit=45, p=0.5),
    A.Perspective(scale=0.05, p=0.2),
    
    # --- Resolution / altitude ---
    A.OneOf([
        A.Downscale(scale_range=(0.5, 0.9), p=1.0),
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0),
    ], p=0.3),
    
    # --- Lighting & shadows ---
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.RandomGamma(gamma_limit=(80, 120), p=0.2),
    A.OneOf([
        A.PlasmaShadow(p=1.0),
        A.RandomShadow(num_shadows_limit=(1, 2), shadow_dimension=5, p=1.0),
    ], p=0.2),
    
    # --- Color ---
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
    A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.2),
    A.CLAHE(clip_limit=4.0, p=0.2),
    
    # --- Noise & compression ---
    A.OneOf([
        A.GaussNoise(std_range=(0.02, 0.06), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.1, 0.3), p=1.0),
        A.ImageCompression(quality_range=(60, 90), p=1.0),
    ], p=0.2),
    
    # --- Occlusion ---
    A.CoarseDropout(num_holes_range=(3, 8), hole_height_range=(8, 25), hole_width_range=(8, 25), p=0.15),
    
    # --- Normalize (always last) ---
    A.Normalize(p=1.0),
]

print('Proposed pipeline:')
for t in proposed_pipeline:
    print(f'  {t.__class__.__name__}(p={t.p})')

# Show samples (without normalize for visualization)
viz_pipeline = [t for t in proposed_pipeline if not isinstance(t, A.Normalize)]
show_augmented('Proposed Combined Pipeline', viz_pipeline, crop_img, crop_kps)


## Summary: What to add vs what we have

| Category | Current | Proposed Addition | Why |
|----------|---------|-------------------|-----|
| **Geometric** | Flip, Rotate90, ShiftScaleRotate, Perspective | scale_limit 0.1→0.15 | Wider altitude range |
| **Blur** | MotionBlur | + GaussianBlur, Sharpen, Downscale (OneOf) | Focus/altitude variation |
| **Lighting** | BrightnessContrast | + RandomGamma, PlasmaShadow | Rock shadows, clouds |
| **Color** | HSV, CLAHE | + RGBShift | Camera white balance |
| **Noise** | — | + GaussNoise/ISONoise/Compression (OneOf) | Sensor variation |
| **Occlusion** | — | + CoarseDropout | Partial visibility |
| **Mosaic** | — | Manual copy-paste (future) | Scene diversity with only 19 images |

### Key principles for point detection:
1. **Never destroy the point**: augmentations must be keypoint-compatible
2. **Preserve local structure**: the ~20-40px region around each iguana must remain recognizable
3. **Don't over-augment**: keep individual augmentation probabilities low (0.15-0.3)
4. **Use OneOf groups**: prevents stacking multiple degradations
5. **Normalize always last**: after all visual augmentations